In [ ]:
#!pip install unpywall selenium requests

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------------- -------------------------- 3.1/9.5 MB 18.5 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 16.0 MB/s  0:00:00
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
  Created wheel for unpywall: filename=unpywall-0.2.3-py3-none-any.whl size=12381 sha256=91a782160467324c9ae170c4dbd7236285fe392c853acfaa1502cf9e7e6c65b8
  Stored in directory: c:\u

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.7 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#!pip install webdriver-manager


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#!pip install undetected-chromedriver

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47214 sha256=ff2c18719dc96f2771df36cac5a5e2c3e6849592cc94e2b815218c337500fe82
  Stored in directory: c:\users\olagunju\appdata\local\pip\cache\wheels\c4\f1\aa\9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected-chromedriver



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Download all open acess and MDPI

In [15]:
"""
PDF Downloader — Open Access Only
=================================
Downloads freely available open-access PDFs. No institutional proxy,
no eID, no password, no Duo.

Two passes:
  Pass 1 — Unpaywall: free OA PDFs via requests (no browser needed)
  Pass 2 — MDPI via Chrome: MDPI is fully open access but sits behind
           Cloudflare, which blocks plain requests. A real browser gets
           through. Chrome is launched only if MDPI DOIs remain, and it
           visits mdpi.com directly (no proxy, no login).

Requirements:
    pip install unpywall requests selenium webdriver-manager

Usage:
    1. Place DOI lists (.txt files) in doi_files/
    2. Run. Chrome opens only for the MDPI pass; leave it alone until done.
"""

import os
import re
import time
import warnings
import requests
from unpywall import Unpywall
from unpywall.utils import UnpywallCredentials

warnings.filterwarnings("ignore")

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

UNPAYWALL_EMAIL = "olagunju@ksu.edu"

DOI_FOLDER    = "doi_files"
OUTPUT_FOLDER = "PDF_files"
LOG_SUCCESS   = "OpenAccess_downloads.txt"
LOG_FAILED    = "ClosedAccess_failed.txt"

DOWNLOAD_WAIT = 20   # seconds to wait for a file to appear after navigation
PAGE_LOAD     = 4    # seconds to wait for a page to load

# ─── PASS 1: UNPAYWALL ────────────────────────────────────────────────────────

def try_unpaywall(doi, output_folder):
    """
    Check Unpaywall for a free legal PDF via direct requests.
    Tries /pdf suffix fallback for MDPI-style URLs.
    """
    try:
        UnpywallCredentials(UNPAYWALL_EMAIL)
        pdf_url = Unpywall.get_pdf_link(doi=doi)
        if not pdf_url:
            return False
        for url in [pdf_url, pdf_url.rstrip("/") + "/pdf"]:
            try:
                r = requests.get(url, timeout=30, verify=False, allow_redirects=True)
                if r.status_code == 200 and b"%PDF" in r.content[:10]:
                    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
                    with open(os.path.join(output_folder, safe_name), "wb") as f:
                        f.write(r.content)
                    print(f"  [UNPAYWALL] Downloaded: {doi}")
                    return True
            except:
                continue
    except Exception as e:
        print(f"  [UNPAYWALL] Error: {doi} - {e}")
    return False


# ─── FILE DETECTION ───────────────────────────────────────────────────────────

def wait_for_new_pdf(output_folder, before_set, seconds=20):
    """
    Watch the output folder for a new completed PDF to appear.
    Ignores .crdownload files (Chrome partial downloads).
    Returns the filename if found, None if timeout.
    """
    deadline = time.time() + seconds
    while time.time() < deadline:
        current = set(os.listdir(output_folder))
        new     = current - before_set
        pdfs    = [f for f in new
                   if f.endswith(".pdf") and not f.endswith(".crdownload")]
        if pdfs:
            return pdfs[0]
        time.sleep(1)
    return None


def rename_to_doi(filename, doi, output_folder):
    """Rename Chrome's auto-generated filename to the DOI-based name."""
    old_path  = os.path.join(output_folder, filename)
    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
    new_path  = os.path.join(output_folder, safe_name)
    if old_path != new_path:
        os.rename(old_path, new_path)
    return new_path


# ─── PASS 2: BROWSER (MDPI ONLY) ──────────────────────────────────────────────

def build_driver(download_folder):
    """
    Configure Chrome to auto-save PDFs to the output folder.
    No proxy and no credentials. Used only for open-access MDPI articles.
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    abs_folder = os.path.abspath(download_folder)
    prefs = {
        "download.default_directory"         : abs_folder,
        "download.prompt_for_download"       : False,
        "plugins.always_open_pdf_externally"  : True,
        "download.directory_upgrade"         : True,
        "safebrowsing.enabled"               : True,
    }

    options = Options()
    options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(30)
    return driver


def resolve_doi(doi):
    """Resolve a DOI to the real publisher URL."""
    try:
        r = requests.head(f"https://doi.org/{doi}",
                          allow_redirects=True, timeout=15, verify=False)
        return r.url
    except Exception as e:
        print(f"  [ERROR] Cannot resolve {doi}: {e}")
        return None


def download_mdpi(driver, doi, real_url, output_folder):
    """
    MDPI strategy:
    MDPI is open access. Navigate Chrome (no proxy) to the article page,
    then to the /pdf URL. MDPI blocks requests but not real browsers.
    """
    pdf_url = real_url.rstrip("/") + "/pdf"
    before  = set(os.listdir(output_folder))

    try:
        driver.get(real_url)
        time.sleep(PAGE_LOAD)
    except:
        pass

    try:
        driver.get(pdf_url)
    except:
        pass

    filename = wait_for_new_pdf(output_folder, before, DOWNLOAD_WAIT)
    if filename:
        rename_to_doi(filename, doi, output_folder)
        return True
    return False


def run_mdpi_pass(mdpi_dois, output_folder):
    """
    Launch Chrome once and work through the MDPI DOIs that Unpaywall missed.
    Returns the list of DOIs successfully downloaded.
    """
    recovered = []
    driver = None
    try:
        driver = build_driver(output_folder)
        for doi in mdpi_dois:
            real_url = resolve_doi(doi)
            if not real_url:
                print(f"  [FAILED] {doi}")
                continue
            if "mdpi.com" not in real_url:
                print(f"  [SKIP] {doi} resolved off-site: {real_url}")
                continue
            if download_mdpi(driver, doi, real_url, output_folder):
                print(f"  [OK] Saved: {doi}")
                recovered.append(doi)
            else:
                print(f"  [FAILED] {doi}")
            time.sleep(2)
    except Exception as e:
        print(f"  [MDPI PASS] Browser error: {e}")
    finally:
        if driver:
            driver.quit()
    return recovered


# ─── UTILITIES ────────────────────────────────────────────────────────────────

def extract_doi_from_line(line):
    line = line.strip()
    if not line or line.lower().startswith("doi"):
        return None
    if "\t" in line:
        line = line.split("\t")[0].strip()
    if line.startswith("http"):
        match = re.search(r'(10\.\d{4,}/\S+)', line)
        return match.group(1) if match else None
    if line.startswith("10."):
        return line
    return None


def load_dois(doi_folder):
    all_dois = []
    for fname in sorted(os.listdir(doi_folder)):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(doi_folder, fname), "r") as f:
            dois = [d for d in (extract_doi_from_line(l) for l in f) if d]
        print(f"  Loaded {len(dois)} DOIs from {fname}")
        all_dois.extend(dois)
    return list(dict.fromkeys(all_dois))


def already_downloaded(doi, output_folder):
    safe_name = doi.replace("/", "_").replace(".", "-") + ".pdf"
    return os.path.exists(os.path.join(output_folder, safe_name))


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(DOI_FOLDER, exist_ok=True)

    dois = load_dois(DOI_FOLDER)
    print(f"\nTotal unique DOIs: {len(dois)}\n")

    success, failed = [], []

    # ── Pass 1: Unpaywall ─────────────────────────────────────────────────────
    print("=== PASS 1: Unpaywall (open access, no browser) ===")
    for doi in dois:
        if already_downloaded(doi, OUTPUT_FOLDER):
            print(f"  [SKIP] Already downloaded: {doi}")
            success.append(doi)
            continue
        if try_unpaywall(doi, OUTPUT_FOLDER):
            success.append(doi)
        else:
            failed.append(doi)

    # ── Pass 2: MDPI via Chrome ───────────────────────────────────────────────
    mdpi_failed = [d for d in failed if d.startswith("10.3390/")]
    if mdpi_failed:
        print(f"\n=== PASS 2: MDPI via Chrome ({len(mdpi_failed)} DOIs) ===")
        print("NOTE: Chrome will open. Leave it alone until the pass finishes.\n")
        recovered = run_mdpi_pass(mdpi_failed, OUTPUT_FOLDER)
        for doi in recovered:
            success.append(doi)
            failed.remove(doi)

    # ── Logs ──────────────────────────────────────────────────────────────────
    with open(LOG_SUCCESS, "w") as f:
        f.write("\n".join(success))
    with open(LOG_FAILED, "w") as f:
        f.write("\n".join(failed))

    print(f"\n{'='*60}")
    print(f"=== FINAL SUMMARY ===")
    print(f"{'='*60}")
    print(f"  Total DOIs processed    : {len(dois)}")
    print(f"  Open-access downloaded  : {len(success)}")
    print(f"  Not open access         : {len(failed)}")
    if dois:
        print(f"  Success rate            : {len(success)/len(dois)*100:.1f}%")

    if failed:
        print(f"\n--- DOIs with no open-access copy found ({len(failed)}) ---")
        for i, doi in enumerate(failed, 1):
            print(f"  {i:3}. {doi}")
        print(f"\n  These have been saved to: {LOG_FAILED}")
        print(f"  Options:")
        print(f"    1. Institutional access : lib.k-state.edu (log in manually)")
        print(f"    2. ILL request          : lib.k-state.edu/interlibrary-loan")
        print(f"    3. Email authors        : search corresponding author on Google Scholar")
        print(f"{'='*60}")


if __name__ == "__main__":
    run()

  Loaded 230 DOIs from Relevant_DOI_250_ARTICLES.txt

Total unique DOIs: 230

=== PASS 1: Unpaywall (open access, no browser) ===
  [SKIP] Already downloaded: 10.3390/foods10030667
  [SKIP] Already downloaded: 10.1111/1750-3841.16467
  [SKIP] Already downloaded: 10.1007/s13197-019-03785-8
  [SKIP] Already downloaded: 10.1111/j.1750-3841.2006.00156.x
  [SKIP] Already downloaded: 10.1007/s13197-020-04241-8
  [SKIP] Already downloaded: 10.1007/s11694-023-01889-6
  [SKIP] Already downloaded: 10.1007/s11694-013-9168-x
  [SKIP] Already downloaded: 10.1111/jfpp.12507
  [SKIP] Already downloaded: 10.1038/s41598-026-35526-1
  [SKIP] Already downloaded: 10.3136/fstr.21.95
  [SKIP] Already downloaded: 10.1002/fsn3.4511
  [SKIP] Already downloaded: 10.1590/fst.03918
  [SKIP] Already downloaded: 10.1111/jfpp.12488
  [SKIP] Already downloaded: 10.3390/foods11192954
  [SKIP] Already downloaded: 10.3390/macromol5010003
  [SKIP] Already downloaded: 10.1111/ijfs.16923
  [SKIP] Already downloaded: 10.100

In [ ]:
! pip install wiley-tdm

  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: c:\Users\olagunju\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [5]:
import sys
print(sys.executable)

c:\Users\olagunju\AppData\Local\anaconda3\envs\Shola-environment\python.exe


In [6]:
%pip install wiley-tdm

  Using cached wiley_tdm-1.2.0-py3-none-any.whl.metadata (15 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached wiley_tdm-1.2.0-py3-none-any.whl (19 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)

  Attempting uninstall: requests

    Found existing installation: requests 2.32.3

    Uninstalling requests-2.32.3:

      Successfully uninstalled requests-2.32.3

   ---------------------------------------- 0/2 [requests]
   -------------------- ------------------- 1/2 [wiley-tdm]
   ---------------------------------------- 2/2 [wiley-tdm]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os
print(os.getcwd())

c:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\DOWNLOAD ARTICLES


In [7]:
from wiley_tdm import TDMClient, DownloadStatus

In [ ]:
"""
Wiley TDM Downloader (official client wrapper)
==============================================
Thin wrapper around WileyLabs' own `wiley-tdm` package. The client does the
API work, pacing, retries and results CSV. This script adds the one thing it
does not do: FILTER YOUR DOI LIST TO WILEY DOIs ONLY.

That filter matters. download_pdfs() attempts every DOI you hand it, so
pointing it straight at download_failed.txt would grind through 114 Elsevier
DOIs at 10 seconds each for guaranteed failures. This script passes only the
10.1111 and 10.1002 DOIs.

HOW ACCESS WORKS (two separate checks):
  1. The token proves you accepted Wiley's TDM license. Get one with your
     ORCID iD at:
       https://onlinelibrary.wiley.com/library-info/resources/text-and-datamining
     Ask K-State Libraries first whether an institutional Wiley TDM agreement
     already exists, since its terms may supersede the click-through license.
  2. Your K-State subscription is checked by IP ADDRESS, not by eID login.
     Run this on the campus network or the K-State VPN. Your eID, password
     and Duo are not used by this API.

SETUP:
    python3 -m venv venv
    source venv/bin/activate          # Windows: venv\\Scripts\\activate
    pip install wiley-tdm

    

  

USAGE:
    python wiley_download.py                    # reads download_failed.txt
    python wiley_download.py my_dois.txt        # or any DOI list
"""

import os
import re
import sys
import shutil
from pathlib import Path

from wiley_tdm import TDMClient, DownloadStatus

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

INPUT_FILE   = "ClosedAccess_failed.txt"
WILEY_DIR    = Path("wiley_downloads")   # where the client writes
FINAL_DIR    = Path("PDF_files")         # your main corpus folder
RESULTS_CSV  = "wiley_results.csv"

PREFIXES = ("10.1111/", "10.1002/")

# Wiley's docs recommend 10 seconds for sustained use (60 requests per
# 10 minutes). The package default is 5.0. Do not lower this: aggressive
# downloading gets institutional IP ranges suspended, which would cut off
# Wiley access for everyone at K-State.
RATE_LIMIT = 10.0

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def pick_input_file():
    """
    Return the DOI list to read.

    In a terminal, sys.argv[1] is the filename you passed. In Jupyter,
    sys.argv[1] is the kernel's own argument, which must be ignored.
    """
    for arg in sys.argv[1:]:
        if arg.startswith("-") or arg.endswith(".json"):
            continue
        if os.path.exists(arg):
            return arg
    return INPUT_FILE


def extract_doi(line):
    line = line.strip()
    if not line or line.lower().startswith("doi"):
        return None
    if "\t" in line:
        line = line.split("\t")[0].strip()
    if line.startswith("http"):
        m = re.search(r'(10\.\d{4,}/\S+)', line)
        return m.group(1) if m else None
    return line if line.startswith("10.") else None


def load_dois(path):
    with open(path) as f:
        dois = [d for d in (extract_doi(l) for l in f) if d]
    return list(dict.fromkeys(dois))


def corpus_name(doi):
    """The naming convention used by your other download scripts."""
    return doi.replace("/", "_").replace(".", "-") + ".pdf"


def already_in_corpus(doi):
    return (FINAL_DIR / corpus_name(doi)).exists()


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    if not os.environ.get(TDMClient.API_TOKEN_ENV):
        print(f"{TDMClient.API_TOKEN_ENV} is not set.")
        print("Get a token with your ORCID iD at:")
        print("  https://onlinelibrary.wiley.com/library-info/resources/text-and-datamining")
        print("\nThen, in a terminal:")
        print(f"    export {TDMClient.API_TOKEN_ENV}='43e170e6-0408-4461-a0d5-02d783b28366'")
        print("Or in a Jupyter cell:")
        print(f"    import os; os.environ['{TDMClient.API_TOKEN_ENV}'] = '43e170e6-0408-4461-a0d5-02d783b28366'")
        return

    path = pick_input_file()
    if not os.path.exists(path):
        print(f"Input file not found: {path}")
        print(f"Working directory is: {os.getcwd()}")
        return

    FINAL_DIR.mkdir(exist_ok=True)
    WILEY_DIR.mkdir(exist_ok=True)

    all_dois = load_dois(path)
    wiley    = [d for d in all_dois if d.startswith(PREFIXES)]
    todo     = [d for d in wiley if not already_in_corpus(d)]

    print(f"Read {len(all_dois)} DOIs from {path}")
    print(f"Wiley DOIs (10.1111, 10.1002): {len(wiley)}")
    if len(wiley) != len(todo):
        print(f"Already in {FINAL_DIR}/: {len(wiley) - len(todo)}")

    if not todo:
        print("Nothing to do.")
        return

    print(f"To download: {len(todo)}")
    print(f"Estimated time at {RATE_LIMIT}s per request: "
          f"{len(todo) * RATE_LIMIT / 60:.1f} minutes\n")

    try:
        tdm = TDMClient(download_dir=WILEY_DIR)
    except ValueError as e:
        print(f"Could not create client: {e}")
        print("The token must be a valid UUID, e.g. 1234abcd-56ef-...")
        return

    tdm.api_rate_limit = RATE_LIMIT

    def progress(result):
        n = len(tdm.results)
        mark = "OK    " if result.status == DownloadStatus.SUCCESS else result.status.name
        size = f"  ({result.size // 1024} KB)" if result.size else ""
        print(f"[{n}/{len(todo)}] {mark:14} {result.doi}{size}")

    results = tdm.download_pdfs(todo, on_result=progress)
    tdm.save_results(RESULTS_CSV)

    # Move successes into the main corpus folder under the shared naming
    # convention, so your other scripts see them as already downloaded.
    moved = 0
    for r in results:
        if r.status == DownloadStatus.SUCCESS and r.path:
            src = Path(r.path)
            if src.exists():
                shutil.move(str(src), str(FINAL_DIR / corpus_name(r.doi)))
                moved += 1

    # Summarise by status so failures are diagnosable at a glance
    counts = {}
    for r in results:
        counts[r.status.name] = counts.get(r.status.name, 0) + 1

    print(f"\n{'='*60}")
    print(f"  Attempted : {len(results)}")
    for status, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"    {status:16} {n}")
    print(f"  Moved into {FINAL_DIR}/ : {moved}")
    print(f"  Full log  : {RESULTS_CSV}")

    if counts.get("ACCESS_DENIED"):
        print("\n  ACCESS_DENIED usually means one of:")
        print("    - you are not on a K-State IP (connect to campus Wi-Fi or VPN)")
        print("    - K-State does not subscribe to that specific journal")
        print("    - the token is not active yet")
        print("  Wiley's check: compare the IP in the client log against the IP")
        print("  your browser shows on Wiley Online Library. If they differ, ask")
        print("  your network admin. If they match, contact tdm@wiley.com.")
    print(f"{'='*60}")


if __name__ == "__main__":
    run()

Read 171 DOIs from download_failed.txt
Wiley DOIs (10.1111, 10.1002): 25
To download: 25
Estimated time at 10.0s per request: 4.2 minutes

[1/25] OK             10.1111/ijfs.16923  (1 KB)


DOI: 10.1111/j.1365-2621.1983.tb14882.x - Access Denied


[2/25] ACCESS_DENIED  10.1111/j.1365-2621.1983.tb14882.x
[3/25] OK             10.1111/ijfs.13458  (0 KB)
[4/25] OK             10.1111/ijfs.15446  (4 KB)
[5/25] OK             10.1111/ijfs.13271  (0 KB)


DOI: 10.1111/j.1365-2621.1986.tb13096.x - Access Denied


[6/25] ACCESS_DENIED  10.1111/j.1365-2621.1986.tb13096.x
[7/25] OK             10.1002/jsfa.70829  (2 KB)
[8/25] OK             10.1111/ijfs.14969  (0 KB)
[9/25] OK             10.1111/ijfs.17244  (0 KB)
[10/25] OK             10.1111/ijfs.13990  (1 KB)
[11/25] OK             10.1002/jsfa.2334  (0 KB)
[12/25] OK             10.1002/cche.10589  (1 KB)
[13/25] OK             10.1111/jfpe.14243  (2 KB)
[14/25] OK             10.1111/jfpe.14578  (2 KB)
[15/25] OK             10.1111/ijfs.17596  (1 KB)
[16/25] OK             10.1111/ijfs.15831  (1 KB)
[17/25] OK             10.1111/1750-3841.14770  (1 KB)
[18/25] OK             10.1111/ijfs.17242  (0 KB)
[19/25] OK             10.1111/jfpp.16764  (1 KB)
[20/25] OK             10.1111/jfpp.13524  (0 KB)
[21/25] OK             10.1111/j.1365-2621.2005.tb09023.x  (0 KB)
[22/25] OK             10.1111/1750-3841.17607  (1 KB)
[23/25] OK             10.1002/jsfa.13233  (0 KB)


DOI: 10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q - Unknown Doi


[24/25] UNKNOWN_DOI    10.1002/(SICI)1521-3803(199808)42:03/04<245::AID-FOOD245>3.3.CO;2-Q
[25/25] OK             10.1111/jfpp.16218  (0 KB)

  Attempted : 25
    SUCCESS          22
    ACCESS_DENIED    2
    UNKNOWN_DOI      1
  Moved into PDF_files/ : 22
  Full log  : wiley_results.csv

  ACCESS_DENIED usually means one of:
    - you are not on a K-State IP (connect to campus Wi-Fi or VPN)
    - K-State does not subscribe to that specific journal
    - the token is not active yet
  Wiley's check: compare the IP in the client log against the IP
  your browser shows on Wiley Online Library. If they differ, ask
  your network admin. If they match, contact tdm@wiley.com.


In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

True